# Clausius + CS Hybrid Scan

This notebook runs a clean `QMM` hybrid model with:

- Clausius attractive mean field
- Carnahan-Starling excluded volume
- symmetric ground-state fit
- liquid critical point
- pure hadronic EOS table
- asymmetric `a_n, a_{pn}, b_n, b_{pn}` fit
- fixed-`y` asymmetric quarkyonic profiles

The hybrid model name inside `QMM` is `clausius_cs`.


## Hybrid Model

The model used here is:

\[
U(n) = -
rac{n}{1 + c n}
\]

for the attractive mean field, together with the Carnahan-Starling repulsive mapping

\[
f_{\mathrm{CS}}(x) = \exp\left[-
rac{3x}{4-x} - 
rac{4x}{(4-x)^2}
ight], \qquad x = bn.
\]

So this is not the original pure Clausius model and not the original pure CS model. It is a new hybrid inside `QMM` called `clausius_cs`.


In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def locate_root() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'src' / 'qmm').exists():
            return candidate.resolve()
    raise RuntimeError('Could not locate the QMM project root from this notebook.')


ROOT = locate_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qmm.config import load_run_config
from qmm.runner import run_workflow

BASE_CONFIG_PATH = ROOT / 'examples' / 'clausius_cs_hybrid_base.json'
OUTPUT_ROOT = ROOT / 'results' / 'generated' / 'clausius_cs_hybrid_scan'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

with BASE_CONFIG_PATH.open('r', encoding='utf-8') as handle:
    base_config = json.load(handle)

print('ROOT =', ROOT)
print('Base config =', BASE_CONFIG_PATH)
print(json.dumps(base_config, indent=2))


## Scan Inputs

The default scan uses the requested endpoint incompressibilities `K0 = [240, 320] MeV`.
Replace the list below if you want a denser scan.


In [ ]:
# Example denser scan:
# target_k0_values = [240.0, 260.0, 280.0, 300.0, 320.0]
target_k0_values = [240.0, 320.0]

proton_fraction_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
lambda_momentum_mev = 300.0
use_quick_quantum = True
write_plots = True

print('K0 targets:', target_k0_values)
print('fixed-y values:', proton_fraction_values)
print('lambda [MeV]:', lambda_momentum_mev)
print('quantum.quick =', use_quick_quantum)


## Build One Config Per Target

This helper starts from the executable base JSON and only overrides the run-specific pieces.


In [ ]:
def build_config_for_target_k0(target_k0: float) -> dict:
    config = deepcopy(base_config)
    run_slug = f'clausius_cs_k0_{int(round(target_k0))}'

    config['run_name'] = run_slug
    config['model']['name'] = 'clausius_cs'
    config['model']['parameter_search']['enabled'] = True
    config['model']['parameter_search']['target_k0'] = float(target_k0)
    config['model']['parameter_search']['parameter_min'] = 0.0
    config['model']['parameter_search']['parameter_max'] = 6.0
    config['model']['parameter_search']['scan_steps'] = 121

    config['workflows']['ground_state'] = True
    config['workflows']['quantum_critical'] = True
    config['workflows']['hadronic_eos_table'] = True
    config['workflows']['asymmetric_fit'] = True
    config['workflows']['asymmetric_quarkyonic'] = True
    config['workflows']['symmetric_quarkyonic'] = False
    config['workflows']['neutron_star'] = False

    config.setdefault('quantum', {})
    config['quantum']['quick'] = bool(use_quick_quantum)

    config['quarkyonic']['lambda_momentum_mev'] = float(lambda_momentum_mev)
    config['asymmetric']['enabled'] = True
    config['asymmetric']['branch_mode'] = 'target_l'
    config['asymmetric']['target_j'] = 32.5
    config['asymmetric']['target_l'] = 58.9
    config['asymmetric']['target_k0'] = float(target_k0)
    config['asymmetric']['proton_fraction_values'] = list(proton_fraction_values)
    config['asymmetric']['beta_equilibrium'] = False

    run_dir = OUTPUT_ROOT / f'K0_{int(round(target_k0))}'
    config['output']['directory'] = str(run_dir)
    config['output']['write_json'] = True
    config['output']['write_csv'] = True
    config['output']['write_plots'] = bool(write_plots)

    return config


## Run The Scan

This cell writes one JSON file per target `K0`, runs the workflow, and stores the returned summaries.


In [ ]:
scan_summaries = {}
failures = []
config_dir = OUTPUT_ROOT / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

for target_k0 in target_k0_values:
    config_dict = build_config_for_target_k0(target_k0)
    config_path = config_dir / f'clausius_cs_k0_{int(round(target_k0))}.json'
    config_path.write_text(json.dumps(config_dict, indent=2), encoding='utf-8')

    try:
        run_config = load_run_config(config_path)
        summary = run_workflow(run_config)
        scan_summaries[target_k0] = summary
        print(f'finished K0={target_k0:.1f} MeV -> c={summary["ground_state"]["parameter_value"]:.6f}')
    except Exception as exc:
        failures.append((target_k0, str(exc)))
        print(f'failed K0={target_k0:.1f} MeV -> {exc}')

if failures:
    print('failures detected:')
    for item in failures:
        print(item)
else:
    print('all runs completed')


## Collect Summaries

The next cell extracts the main outputs: the resolved Clausius parameter `c`, the symmetric `a,b`, the liquid critical point, and the asymmetric couplings.


In [ ]:
ground_rows = []
quantum_rows = []
asym_rows = []

for target_k0, summary in sorted(scan_summaries.items()):
    ground = dict(summary['ground_state'])
    ground['target_k0_requested'] = target_k0
    ground_rows.append(ground)

    quantum = dict(summary['quantum_critical'])
    quantum['target_k0_requested'] = target_k0
    quantum_rows.append(quantum)

    asym = dict(summary['asymmetric_fit'])
    asym['target_k0_requested'] = target_k0
    asym_rows.append(asym)

ground_df = pd.DataFrame(ground_rows).sort_values('target_k0_requested').reset_index(drop=True)
quantum_df = pd.DataFrame(quantum_rows).sort_values('target_k0_requested').reset_index(drop=True)
asym_df = pd.DataFrame(asym_rows).sort_values('target_k0_requested').reset_index(drop=True)

ground_df.to_csv(OUTPUT_ROOT / 'ground_state_summary.csv', index=False)
quantum_df.to_csv(OUTPUT_ROOT / 'quantum_critical_summary.csv', index=False)
asym_df.to_csv(OUTPUT_ROOT / 'asymmetric_fit_summary.csv', index=False)

display(ground_df[['target_k0_requested', 'parameter_value', 'a', 'b', 'K0', 'n_sat', 'binding']])
display(quantum_df[['target_k0_requested', 'Tc', 'nc', 'Pc', 'iterations', 'score']])
display(asym_df[['target_k0_requested', 'a_n', 'a_pn', 'b_n', 'b_pn', 'ratio_a_pn_over_a_n', 'ratio_b_pn_over_b_n', 'J', 'L']])


## Ground-State And Critical-Point Plots


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

axes[0, 0].plot(ground_df['target_k0_requested'], ground_df['parameter_value'], marker='o', lw=2)
axes[0, 0].set_xlabel(r'target $K_0$ [MeV]')
axes[0, 0].set_ylabel('resolved c')
axes[0, 0].set_title('Clausius parameter from target K0')

axes[0, 1].plot(ground_df['target_k0_requested'], ground_df['a'], marker='o', lw=2, label='a')
axes[0, 1].plot(ground_df['target_k0_requested'], ground_df['b'], marker='s', lw=2, label='b')
axes[0, 1].set_xlabel(r'target $K_0$ [MeV]')
axes[0, 1].set_title('Symmetric nuclear-matter couplings')
axes[0, 1].legend(loc='best')

axes[1, 0].scatter(quantum_df['nc'], quantum_df['Tc'], c=quantum_df['target_k0_requested'], cmap='viridis', s=120)
for row in quantum_df.itertuples():
    axes[1, 0].annotate(f"K0={row.target_k0_requested:.0f}", (row.nc, row.Tc), xytext=(5, 5), textcoords='offset points', fontsize=8)
axes[1, 0].set_xlabel(r'$n_c$ [fm$^{-3}$]')
axes[1, 0].set_ylabel(r'$T_c$ [MeV]')
axes[1, 0].set_title(r'$T_c$ vs $n_c$')

axes[1, 1].plot(quantum_df['target_k0_requested'], quantum_df['Tc'], marker='o', lw=2, label=r'$T_c$')
axes[1, 1].plot(quantum_df['target_k0_requested'], quantum_df['nc'], marker='s', lw=2, label=r'$n_c$')
axes[1, 1].set_xlabel(r'target $K_0$ [MeV]')
axes[1, 1].set_title('Critical-point evolution')
axes[1, 1].legend(loc='best')

plt.show()


## Pure Hadronic EOS Table

Each run writes a dense zero-temperature hadronic EOS table before quarkyonic matter is introduced. The CSV includes the reconstructed hadronic sound speed.


In [ ]:
selected_target_k0_hadronic = target_k0_values[0]
hadronic_summary = scan_summaries[selected_target_k0_hadronic]
hadronic_csv = Path(hadronic_summary['outputs']['hadronic_eos_csv'])
hadronic_df = pd.read_csv(hadronic_csv)

print('Hadronic EOS CSV:', hadronic_csv)
display(hadronic_df.head())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)

axes[0].plot(hadronic_df['n_over_n0'], hadronic_df['e_per_particle_minus_m'], lw=2.2)
axes[0].axhline(0.0, color='black', ls='--', lw=1.0)
axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$E/A - m_N$ [MeV]')
axes[0].set_title(f'Hadronic binding curve, K0={selected_target_k0_hadronic:.0f} MeV')

axes[1].plot(hadronic_df['n_over_n0'], hadronic_df['pressure'], lw=2.2)
axes[1].axhline(0.0, color='black', ls='--', lw=1.0)
axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$P$ [MeV fm$^{-3}$]')
axes[1].set_title('Pure hadronic pressure')

axes[2].plot(hadronic_df['n_over_n0'], hadronic_df['vs2'], lw=2.2)
axes[2].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
axes[2].set_ylim(-0.1, 0.8)
axes[2].set_xlabel(r'$n/n_0$')
axes[2].set_ylabel(r'$v_s^2$')
axes[2].set_title('Pure hadronic sound speed')

plt.show()


## Asymmetric Couplings

This is where the notebook summarizes `a_n`, `a_{pn}`, `b_n`, `b_{pn}`, and the split ratios.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)

axes[0].plot(asym_df['target_k0_requested'], asym_df['a_n'], marker='o', lw=2, label=r'$a_n$')
axes[0].plot(asym_df['target_k0_requested'], asym_df['a_pn'], marker='s', lw=2, label=r'$a_{pn}$')
axes[0].set_xlabel(r'target $K_0$ [MeV]')
axes[0].set_ylabel(r'[MeV fm$^3$]')
axes[0].set_title('Attractive asymmetric couplings')
axes[0].legend(loc='best')

axes[1].plot(asym_df['target_k0_requested'], asym_df['b_n'], marker='o', lw=2, label=r'$b_n$')
axes[1].plot(asym_df['target_k0_requested'], asym_df['b_pn'], marker='s', lw=2, label=r'$b_{pn}$')
axes[1].set_xlabel(r'target $K_0$ [MeV]')
axes[1].set_ylabel(r'[fm$^3$]')
axes[1].set_title('Repulsive asymmetric couplings')
axes[1].legend(loc='best')

plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
axes[0].plot(asym_df['target_k0_requested'], asym_df['ratio_a_pn_over_a_n'], marker='o', lw=2)
axes[0].set_xlabel(r'target $K_0$ [MeV]')
axes[0].set_ylabel(r'$a_{pn}/a_n$')
axes[0].set_title('Attractive split ratio')

axes[1].plot(asym_df['target_k0_requested'], asym_df['ratio_b_pn_over_b_n'], marker='o', lw=2)
axes[1].set_xlabel(r'target $K_0$ [MeV]')
axes[1].set_ylabel(r'$b_{pn}/b_n$')
axes[1].set_title('Repulsive split ratio')

plt.show()


## Fixed-`y` Asymmetric Quarkyonic Profiles

Pick one target `K0` below and the notebook will load all fixed-`y` outputs from that run.


In [ ]:
selected_target_k0 = target_k0_values[0]
selected_summary = scan_summaries[selected_target_k0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

for y_value in proton_fraction_values:
    key = f'asymmetric_y_{y_value:0.3f}_csv'
    csv_path = Path(selected_summary['outputs'][key])
    df = pd.read_csv(csv_path)
    label = f'y={y_value:.1f}'
    axes[0].plot(df['n_over_n0'], df['quark_fraction'], lw=2, label=label)
    axes[1].plot(df['n_over_n0'], df['vs2'], lw=2, label=label)

axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$f_Q$')
axes[0].set_title(f'Quark fraction for K0={selected_target_k0:.0f} MeV')
axes[0].legend(loc='upper left', ncol=2)

axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$v_s^2$')
axes[1].set_ylim(-0.1, 0.8)
axes[1].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
axes[1].set_title(f'Sound speed for K0={selected_target_k0:.0f} MeV')
axes[1].legend(loc='upper right', ncol=2)

plt.show()


## Output Check

This final cell lists the generated JSON inputs and output folders for the scan.


In [ ]:
print('Generated config files:')
for path in sorted((OUTPUT_ROOT / 'configs').glob('*.json')):
    print(' -', path.relative_to(ROOT))

print('Output folders:')
for path in sorted(OUTPUT_ROOT.glob('K0_*')):
    if path.is_dir():
        print(' -', path.relative_to(ROOT))
